# P-Wave Moveout and Ps Conversion — GeoNet / 2011 Tohoku M9

Loads `tohoku_nz_array.npz` (run `download_data.py` first), picks P and Ps arrivals, fits linear moveout curves, and plots a three-component record section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load data

In [ ]:
d = np.load('tohoku_nz_array.npz', allow_pickle=True)

Z         = d['Z']            # (N stations, T samples) — vertical
R         = d['R']            # radial (toward earthquake)
T         = d['T']            # transverse
distances = d['distances']    # epicentral distance, km
times     = d['times_rel']    # seconds relative to predicted P (0 = P)
stations  = d['stations']
sr        = float(d['sampling_rate'])  # samples/sec

print(f"{Z.shape[0]} stations, {Z.shape[1]} samples each")
print(f"Distance range: {distances.min():.0f} – {distances.max():.0f} km")
print(f"Time window:    {times[0]:.0f} to {times[-1]:.0f} s relative to P")

## 2. Sort stations by distance (north → south across NZ)

In [ ]:
order     = np.argsort(distances)
Z         = Z[order]
R         = R[order]
T         = T[order]
distances = distances[order]
stations  = stations[order]

print("Stations (closest first):")
for sta, dist in zip(stations, distances):
    print(f"  {sta}  {dist:.0f} km")

## 3. Quick look — raw traces

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

mid = len(stations) // 2   # pick a station in the middle of the array

for ax, comp, label, col in zip(axes,
                                 [Z[mid], R[mid], T[mid]],
                                 ['Z (vertical)', 'R (radial)', 'T (transverse)'],
                                 ['#2c3e50', '#c0392b', '#16a085']):
    ax.plot(times, comp, color=col, linewidth=0.8)
    ax.set_ylabel(label, fontsize=9)
    ax.axvline(0, color='gray', linewidth=0.8, linestyle=':')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time relative to predicted P (s)')
fig.suptitle(f'Station {stations[mid]}  —  {distances[mid]:.0f} km from Tohoku', fontsize=11)
plt.tight_layout()

## 4. Pick P on vertical (Z) and Ps on radial (R)

In [ ]:
def time_to_idx(t_sec):
    """Convert a time in seconds (relative to P) to a sample index."""
    return int(round((t_sec - times[0]) * sr))

N = len(stations)
p_picks  = np.empty(N)
ps_picks = np.empty(N)

P_WIN  = (0.0,  8.0)    # search for P in this window on Z
PS_WIN = (2.0, 10.0)    # search for Ps relative to picked P, on R

for i in range(N):
    # P: max amplitude on Z
    i0, i1   = time_to_idx(P_WIN[0]), time_to_idx(P_WIN[1])
    seg      = Z[i, i0:i1]
    p_picks[i] = times[i0 + np.argmax(np.abs(seg))]

    # Ps: max amplitude on R, searched after picked P
    j0 = time_to_idx(p_picks[i] + PS_WIN[0])
    j1 = time_to_idx(p_picks[i] + PS_WIN[1])
    j0, j1 = max(0, j0), min(Z.shape[1] - 1, j1)
    seg_r      = R[i, j0:j1]
    ps_picks[i] = times[j0 + np.argmax(np.abs(seg_r))]

tau_ps = ps_picks - p_picks
print(f"P picks  range: {p_picks.min():.1f} – {p_picks.max():.1f} s")
print(f"Ps picks range: {ps_picks.min():.1f} – {ps_picks.max():.1f} s")
print(f"τ_Ps     range: {tau_ps.min():.1f} – {tau_ps.max():.1f} s  (median {np.median(tau_ps):.1f} s)")

## 5. Fit linear moveout to P and Ps

In [ ]:
def linear(dist, a, v):
    """t = a + d/v"""
    return a + dist / v

popt_p,  pcov_p  = curve_fit(linear, distances, p_picks,  p0=[0.0, 8.0])
popt_ps, pcov_ps = curve_fit(linear, distances, ps_picks, p0=[5.0, 8.0])

v_p  = popt_p[1];   v_p_err  = np.sqrt(pcov_p[1, 1])
v_ps = popt_ps[1];  v_ps_err = np.sqrt(pcov_ps[1, 1])

# τ_Ps at the median distance
d_mid  = np.median(distances)
tau_fit = linear(d_mid, *popt_ps) - linear(d_mid, *popt_p)

# Rough crustal thickness from τ_Ps
Vp, Vs, p = 6.4, 3.7, 0.068   # km/s, km/s, s/km
eta_S = np.sqrt(1/Vs**2 - p**2)
eta_P = np.sqrt(1/Vp**2 - p**2)
H_est = tau_fit / (eta_S - eta_P)

print(f"P  apparent velocity : {v_p:.2f} ± {v_p_err:.2f} km/s")
print(f"Ps apparent velocity : {v_ps:.2f} ± {v_ps_err:.2f} km/s")
print(f"τ_Ps (fitted)        : {tau_fit:.2f} s")
print(f"Crustal thickness H  : {H_est:.0f} km")

## 6. Three-component record section

In [ ]:
SCALE   = 60.0    # km — visual half-amplitude
d_line  = np.linspace(distances.min(), distances.max(), 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 10), sharey=True)

comp_data  = [Z,         R,              T]
comp_names = ['Z (vertical)', 'R (radial)', 'T (transverse)']
comp_cols  = ['#2c3e50',   '#c0392b',   '#16a085']

for ax, comp, name, col in zip(axes, comp_data, comp_names, comp_cols):
    for i in range(N):
        ax.plot(times, comp[i] * SCALE + distances[i],
                color=col, linewidth=0.6, alpha=0.75)

    # P picks
    ax.scatter(p_picks, distances, marker='|', s=80,
               linewidths=1.5, color='steelblue', zorder=4, label='P')

    # Ps picks on radial only
    if 'radial' in name:
        ax.scatter(ps_picks, distances, marker='|', s=80,
                   linewidths=1.5, color='gold', zorder=4, label='Ps')

    # Moveout lines
    ax.plot(linear(d_line, *popt_p), d_line,
            color='steelblue', linewidth=1.8, linestyle='--',
            label=f'P  v={v_p:.1f} km/s')
    if 'radial' in name:
        ax.plot(linear(d_line, *popt_ps), d_line,
                color='gold', linewidth=1.8, linestyle='--',
                label=f'Ps τ={tau_fit:.1f} s')

    ax.axvline(0, color='gray', linewidth=0.8, linestyle=':')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Time relative to predicted P (s)', fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(times[0], times[-1])

axes[0].set_ylabel('Epicentral distance (km)', fontsize=11)
axes[0].invert_yaxis()

fig.suptitle('2011 Tohoku M9.0 — GeoNet NZ array  (0.5–2 Hz)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('record_section.png', dpi=150, bbox_inches='tight')

## 7. Moveout fit: P vs Ps

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 9),
                                gridspec_kw={'height_ratios': [3, 1]})

# Top: picks and fits
ax1.scatter(distances, p_picks,  color='steelblue', s=55, zorder=3,
            edgecolors='white', linewidths=0.5, label='P picks  (Z)')
ax1.scatter(distances, ps_picks, color='#e67e22',   s=55, zorder=3,
            edgecolors='white', linewidths=0.5, label='Ps picks (R)')
ax1.plot(d_line, linear(d_line, *popt_p),
         color='steelblue', linewidth=2,
         label=f'P fit  v = {v_p:.2f} km/s')
ax1.plot(d_line, linear(d_line, *popt_ps),
         color='#e67e22', linewidth=2,
         label=f'Ps fit  τ = {tau_fit:.1f} s  →  H ≈ {H_est:.0f} km')

ax1.set_ylabel('Arrival time relative to predicted P (s)', fontsize=11)
ax1.set_title(f'P and Ps Moveout — τ_Ps = {tau_fit:.1f} s  →  H ≈ {H_est:.0f} km', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Bottom: per-station τ_Ps (should be roughly flat)
ax2.scatter(distances, tau_ps, color='#e67e22', s=40,
            edgecolors='white', linewidths=0.4)
ax2.axhline(np.median(tau_ps), color='gray', linewidth=1.5, linestyle='--',
            label=f'Median = {np.median(tau_ps):.1f} s')
ax2.set_xlabel('Epicentral distance (km)', fontsize=11)
ax2.set_ylabel('τ_Ps  (s)', fontsize=11)
ax2.set_title('P-to-Ps delay per station', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('moveout_fit.png', dpi=150, bbox_inches='tight')

## 8. Summary

In [ ]:
print("Results")
print("-------")
print(f"Stations             : {N}")
print(f"P apparent velocity  : {v_p:.2f} ± {v_p_err:.2f} km/s")
print(f"Ps apparent velocity : {v_ps:.2f} ± {v_ps_err:.2f} km/s")
print(f"τ_Ps                 : {tau_fit:.2f} s")
print(f"Crustal thickness H  : {H_est:.0f} km  (assuming Vp={Vp}, Vs={Vs} km/s)")